In [48]:
import numpy as np
import pandas as pd

from collections import defaultdict, deque

pd.set_option('display.max_columns', None)

In [49]:
df = pd.read_csv("../data/raw/1993-94.csv")

In [50]:
# Limpieza
df = df.loc[:, ~df.columns.str.startswith("Unnamed")]
df = df.drop_duplicates()
df = df.dropna(how='all')

In [51]:
# Columnas críticas
df = df.dropna(subset=["HomeTeam", "AwayTeam", "FTHG", "FTAG", "Date"])

In [52]:
# Tipos
df["FTHG"] = df["FTHG"].astype(int)
df["FTAG"] = df["FTAG"].astype(int)

In [53]:
# Fecha + orden
df["Date"] = pd.to_datetime(df["Date"], format="%d/%m/%y")
df = df.sort_values("Date").reset_index(drop=True)

In [54]:
# =========================
# 2. ESTRUCTURAS DE DATOS
# =========================

# Historial últimos 5 partidos (W/D/L)
# Ejemplo: ["W","D","L","W","W"]
hist_results = defaultdict(lambda: deque(maxlen=5))

# Goles marcados últimos 5 partidos
# Ejemplo: [2,1,0,3,2]
hist_goals_for = defaultdict(lambda: deque(maxlen=5))

# Goles encajados últimos 5 partidos
hist_goals_against = defaultdict(lambda: deque(maxlen=5))


# Historial SOLO como local
# Ejemplo: últimos 5 partidos jugando en casa
hist_home = defaultdict(lambda: deque(maxlen=5))

# Historial SOLO como visitante
hist_away = defaultdict(lambda: deque(maxlen=5))


# Head-to-head (enfrentamientos entre 2 equipos)
# Guardamos últimos 3 resultados entre ambos
# Ejemplo: Madrid vs Barca → ["W","L","D"]
h2h = defaultdict(lambda: deque(maxlen=3))


# Totales de goles en la temporada
# Ejemplo: Madrid lleva 30 goles en total
season_gf = defaultdict(int)
season_ga = defaultdict(int)


# ELO rating (fuerza del equipo)
# Empiezan todos en 1500
elo = defaultdict(lambda: 1500)


# Última fecha en que jugó cada equipo
# Sirve para calcular descanso
last_match_date = {}

In [55]:
# =========================
# 3. FUNCIONES AUXILIARES
# =========================

def result_to_points(r):
    """
    Convierte resultado a puntos:
    W = 3, D = 1, L = 0
    """
    return 3 if r == "W" else 1 if r == "D" else 0


def weighted_points(results):
    """
    Calcula puntos ponderados (más peso a lo reciente)

    Ejemplo:
    results = ["W","D","L","W","W"]

    pesos = [1,2,3,4,5]

    resultado:
    = 3*1 + 1*2 + 0*3 + 3*4 + 3*5
    """
    weights = [1, 2, 3, 4, 5]
    return sum(result_to_points(r) * w for r, w in zip(results, weights[-len(results):]))


def elo_expected(rating_a, rating_b):
    """
    Probabilidad de que A gane

    Ejemplo:
    A = 1500, B = 1500 → 0.5
    A = 1700, B = 1500 → > 0.5
    """
    return 1 / (1 + 10 ** ((rating_b - rating_a) / 400))


def update_elo(r_a, r_b, score_a, k=20):
    """
    Actualiza el ELO tras el partido

    score_a:
    ganar = 1
    empate = 0.5
    perder = 0
    """
    exp_a = elo_expected(r_a, r_b)
    return r_a + k * (score_a - exp_a)

In [ ]:
# =========================
# 4. LOOP PRINCIPAL
# =========================

features = []

for _, row in df.iterrows():

    home = row["HomeTeam"]   # equipo local
    away = row["AwayTeam"]   # equipo visitante
    hg = row["FTHG"]         # goles local
    ag = row["FTAG"]         # goles visitante
    date = row["Date"]


    # =========================
    # RESULTADO DEL PARTIDO
    # =========================

    if hg > ag:
        hr, ar = "W", "L"
        score_home = 1
    elif hg < ag:
        hr, ar = "L", "W"
        score_home = 0
    else:
        hr, ar = "D", "D"
        score_home = 0.5


    # =========================
    # FEATURES (ANTES DEL PARTIDO)
    # =========================

    # 🔥 FORMA RECIENTE (últimos 5 partidos)
    home_points_5 = sum(result_to_points(r) for r in hist_results[home])
    away_points_5 = sum(result_to_points(r) for r in hist_results[away])

    # Ejemplo:
    # ["W","W","D","L","W"] → 3+3+1+0+3 = 10


    # 🔥 FORMA PONDERADA (más peso a lo reciente)
    home_weighted = weighted_points(hist_results[home])
    away_weighted = weighted_points(hist_results[away])


    # 🔥 GOLES ÚLTIMOS 5
    home_gf_5 = sum(hist_goals_for[home])
    home_ga_5 = sum(hist_goals_against[home])

    away_gf_5 = sum(hist_goals_for[away])
    away_ga_5 = sum(hist_goals_against[away])


    # 🔥 DIFERENCIA DE GOLES
    home_diff_5 = home_gf_5 - home_ga_5
    away_diff_5 = away_gf_5 - away_ga_5

    # Ejemplo:
    # goles a favor = 10
    # goles en contra = 5
    # diff = +5


    # 🔥 RENDIMIENTO LOCAL / VISITANTE
    home_home_points = sum(result_to_points(r) for r in hist_home[home])
    away_away_points = sum(result_to_points(r) for r in hist_away[away])

    # Ejemplo:
    # Madrid en casa: ["W","W","D"] → 7 puntos


    # 🔥 HEAD TO HEAD
    pair = tuple(sorted([home, away]))
    h2h_points = sum(result_to_points(r) for r in h2h[pair])

    # Ejemplo:
    # últimos enfrentamientos → ["W","L","W"] → 6 puntos


    # 🔥 ELO (fuerza del equipo)
    elo_home = elo[home]
    elo_away = elo[away]

    # Ejemplo:
    # Madrid = 1700, Getafe = 1500


    # 🔥 DESCANSO
    rest_home = (date - last_match_date.get(home, date)).days
    rest_away = (date - last_match_date.get(away, date)).days

    # Ejemplo:
    # si jugó hace 3 días → rest = 3


    # =========================
    # GUARDAR FEATURES
    # =========================

    features.append({
        "home_points_5": home_points_5,
        "away_points_5": away_points_5,

        "home_weighted": home_weighted,
        "away_weighted": away_weighted,

        "home_diff_5": home_diff_5,
        "away_diff_5": away_diff_5,

        "home_home_points": home_home_points,
        "away_away_points": away_away_points,

        "h2h_points": h2h_points,

        "elo_home": elo_home,
        "elo_away": elo_away,

        "rest_home": rest_home,
        "rest_away": rest_away,
    })


    # =========================
    # ACTUALIZAR HISTORIALES (DESPUÉS DEL PARTIDO)
    # =========================

    # Resultados
    hist_results[home].append(hr)
    hist_results[away].append(ar)

    # Goles
    hist_goals_for[home].append(hg)
    hist_goals_for[away].append(ag)

    hist_goals_against[home].append(ag)
    hist_goals_against[away].append(hg)

    # Local / visitante
    hist_home[home].append(hr)
    hist_away[away].append(ar)

    # Head to head
    h2h[pair].append(hr)

    # Temporada
    season_gf[home] += hg
    season_ga[home] += ag

    season_gf[away] += ag
    season_ga[away] += hg


    # 🔥 ACTUALIZAR ELO
    elo[home] = update_elo(elo[home], elo[away], score_home)
    elo[away] = update_elo(elo[away], elo[home], 1 - score_home)


    # Guardar última fecha
    last_match_date[home] = date
    last_match_date[away] = date

In [ ]:

for _, row in df.iterrows():

    home = row["HomeTeam"]   # equipo local
    away = row["AwayTeam"]   # equipo visitante
    hg = row["FTHG"]         # goles local
    ag = row["FTAG"]         # goles visitante
    date = row["Date"]


    # =========================
    # RESULTADO DEL PARTIDO
    # =========================

    if hg > ag:
        hr, ar = "W", "L"
        score_home = 1
    elif hg < ag:
        hr, ar = "L", "W"
        score_home = 0
    else:
        hr, ar = "D", "D"
        score_home = 0.5


    # =========================
    # FEATURES (ANTES DEL PARTIDO)
    # =========================

    # 🔥 FORMA RECIENTE (últimos 5 partidos)
    home_points_5 = sum(result_to_points(r) for r in hist_results[home])
    away_points_5 = sum(result_to_points(r) for r in hist_results[away])

    # Ejemplo:
    # ["W","W","D","L","W"] → 3+3+1+0+3 = 10


    # 🔥 FORMA PONDERADA (más peso a lo reciente)
    home_weighted = weighted_points(hist_results[home])
    away_weighted = weighted_points(hist_results[away])


    # 🔥 GOLES ÚLTIMOS 5
    home_gf_5 = sum(hist_goals_for[home])
    home_ga_5 = sum(hist_goals_against[home])

    away_gf_5 = sum(hist_goals_for[away])
    away_ga_5 = sum(hist_goals_against[away])


    # 🔥 DIFERENCIA DE GOLES
    home_diff_5 = home_gf_5 - home_ga_5
    away_diff_5 = away_gf_5 - away_ga_5

    # Ejemplo:
    # goles a favor = 10
    # goles en contra = 5
    # diff = +5


    # 🔥 RENDIMIENTO LOCAL / VISITANTE
    home_home_points = sum(result_to_points(r) for r in hist_home[home])
    away_away_points = sum(result_to_points(r) for r in hist_away[away])

    # Ejemplo:
    # Madrid en casa: ["W","W","D"] → 7 puntos


    # 🔥 HEAD TO HEAD
    pair = tuple(sorted([home, away]))
    h2h_points = sum(result_to_points(r) for r in h2h[pair])

    # Ejemplo:
    # últimos enfrentamientos → ["W","L","W"] → 6 puntos


    # 🔥 ELO (fuerza del equipo)
    elo_home = elo[home]
    elo_away = elo[away]

    # Ejemplo:
    # Madrid = 1700, Getafe = 1500


    # 🔥 DESCANSO
    rest_home = (date - last_match_date.get(home, date)).days
    rest_away = (date - last_match_date.get(away, date)).days

    # Ejemplo:
    # si jugó hace 3 días → rest = 3


    # =========================
    # GUARDAR FEATURES
    # =========================

    features.append({
        "home_points_5": home_points_5,
        "away_points_5": away_points_5,

        "home_weighted": home_weighted,
        "away_weighted": away_weighted,

        "home_diff_5": home_diff_5,
        "away_diff_5": away_diff_5,

        "home_home_points": home_home_points,
        "away_away_points": away_away_points,

        "h2h_points": h2h_points,

        "elo_home": elo_home,
        "elo_away": elo_away,

        "rest_home": rest_home,
        "rest_away": rest_away,
    })


    # =========================
    # ACTUALIZAR HISTORIALES (DESPUÉS DEL PARTIDO)
    # =========================

    # Resultados
    hist_results[home].append(hr)
    hist_results[away].append(ar)

    # Goles
    hist_goals_for[home].append(hg)
    hist_goals_for[away].append(ag)

    hist_goals_against[home].append(ag)
    hist_goals_against[away].append(hg)

    # Local / visitante
    hist_home[home].append(hr)
    hist_away[away].append(ar)

    # Head to head
    h2h[pair].append(hr)

    # Temporada
    season_gf[home] += hg
    season_ga[home] += ag

    season_gf[away] += ag
    season_ga[away] += hg


    # 🔥 ACTUALIZAR ELO
    elo[home] = update_elo(elo[home], elo[away], score_home)
    elo[away] = update_elo(elo[away], elo[home], 1 - score_home)


    # Guardar última fecha
    last_match_date[home] = date
    last_match_date[away] = date

In [57]:
# =========================
# 5. UNIR FEATURES
# =========================

df_features = pd.DataFrame(features)

df = pd.concat([df, df_features], axis=1)

In [58]:
df[(df["HomeTeam"] == "Coventry") | (df["AwayTeam"] == "Coventry")]

,Div,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,home_points_5,away_points_5,home_weighted,away_weighted,home_diff_5,away_diff_5,home_home_points,away_away_points,h2h_points,elo_home,elo_away,rest_home,rest_away
0,E0,1993-08-14,Arsenal,Coventry,0,3,A,0,0,0,0,0,0,0,0,0,1500.000000,1500.000000,0,0
17,E0,1993-08-18,Coventry,Newcastle,2,1,H,3,0,15,0,3,-1,0,0,0,1509.712256,1490.000000,4,4
29,E0,1993-08-21,Coventry,West Ham,1,1,D,6,0,27,0,4,-3,3,0,0,1519.145500,1480.566756,3,4
36,E0,1993-08-24,Oldham,Coventry,3,3,D,3,7,12,26,-3,4,0,3,0,1489.991982,1518.039656,3,3
44,E0,1993-08-27,Man City,Coventry,1,1,D,1,8,2,24,-4,4,1,4,0,1471.892442,1517.257167,3,3
60,E0,1993-09-01,Coventry,Liverpool,1,0,H,9,12,21,33,4,10,4,6,0,1515.995591,1526.470987,5,4
74,E0,1993-09-11,Aston Villa,Coventry,0,0,D,8,9,30,27,1,2,6,5,0,1519.917911,1526.297006,11,10
83,E0,1993-09-18,Coventry,Chelsea,1,1,D,7,8,23,28,1,2,7,2,0,1526.118704,1503.317668,7,7
96,E0,1993-09-25,Coventry,Leeds,0,2,A,7,9,21,36,1,1,8,4,0,1525.463378,1510.061701,7,7
104,E0,1993-10-02,Norwich,Coventry,1,0,H,6,6,22,14,3,-1,4,6,0,1517.027154,1515.020372,7,7


In [59]:
df.to_csv("../data/93_94.csv", index=None)